# TD3 Self-Driving Car - Google Colab Research Workflow

This notebook is a beginner-friendly, end-to-end Colab workflow for the TD3 car project.

It is compatible with both:
- Google Colab (recommended)
- Local notebook execution (for debugging)

### Workflow Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | **Setup** | Clone repo & install dependencies |
| 2 | **Environment Check** | Verify headless mode, GPU, assets |
| 3 | **Batch 1** | Run experiments 1–3 |
| 4 | **Batch 2** | Run experiments 4–6 |
| 5 | **Batch 3** | Run experiments 7–9 |
| 6 | **Plot Generation** | Generate all plots from logs |
| 7 | **Save Results** | Organize output into `results/` |
| 8 | **Download Results** | Zip and download everything |

> **Tip:** Run batches one at a time to avoid Colab timeouts. Each batch is independent — you can reconnect and continue from where you left off.

---
## 1. Setup — Clone Repo & Install Dependencies

This cell supports both Colab and local execution:
- If the project already exists, it uses it.
- Otherwise, it clones from GitHub.
- Then it switches into the project directory and validates key files.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/adityagangwani30/TD3-Car-Game.git"
DEFAULT_REPO_NAME = "TD3-Car-Game"

def resolve_project_dir() -> Path:
    cwd = Path.cwd()

    # Case 1: already inside the repo
    if (cwd / "main.py").exists() and (cwd / "requirements.txt").exists():
        return cwd

    # Case 2: local workspace has repo folder
    local_candidate = cwd / DEFAULT_REPO_NAME
    if (local_candidate / "main.py").exists():
        return local_candidate

    # Case 3: Colab default path
    colab_base = Path("/content")
    colab_candidate = colab_base / DEFAULT_REPO_NAME
    if colab_candidate.exists():
        return colab_candidate

    # Clone into /content on Colab, otherwise into current working directory
    clone_parent = colab_base if colab_base.exists() else cwd
    target = clone_parent / DEFAULT_REPO_NAME
    if not target.exists():
        print(f"Cloning repository to: {target}")
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    return target

PROJECT_DIR = resolve_project_dir()
os.chdir(PROJECT_DIR)

required_files = ["main.py", "requirements.txt", "run_experiments.py", "environment.py"]
missing = [f for f in required_files if not (PROJECT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files in project directory: {missing}")

print("Current directory:", Path.cwd())
print("Detected project directory:", PROJECT_DIR)
print("Top-level files:", sorted([p.name for p in PROJECT_DIR.iterdir()])[:20])

In [ ]:
import subprocess
import sys

print("Installing/upgrading pip...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Installing project dependencies from requirements.txt...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Import verification
import numpy
import pygame
import torch

print("Dependency check passed")
print("numpy:", numpy.__version__)
print("pygame:", pygame.__version__)
print("torch:", torch.__version__)

---
## 2. Environment Check — Headless Mode & GPU

Colab is headless, so we force SDL to dummy drivers before running simulation code.

This cell also validates:
- GPU availability
- key project imports
- asset generation readiness

In [ ]:
import os
from pathlib import Path

# Force safe headless behavior in Colab/notebook environments
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import torch
from utils import detect_headless_environment, ensure_assets_exist

device = "cuda" if torch.cuda.is_available() else "cpu"
headless_detected = detect_headless_environment()

ensure_assets_exist()

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Using device:", device)
print("Headless detected:", headless_detected)
print("Working directory:", Path.cwd())
print("Assets directory exists:", Path("assets").exists())

# Create output directories upfront
Path("logs").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)
print("Output directories ready (logs/, models/)")

### Quick Sanity Check — Headless Demo

Runs a very short demo to verify the environment works end-to-end. Displays a preview frame if generated.

In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

preview_path = Path("logs/demo_preview.png")
preview_path.parent.mkdir(parents=True, exist_ok=True)

print("Running demo in headless mode...")
subprocess.run([
    sys.executable,
    "main.py",
    "--mode", "demo",
    "--demo-episodes", "2",
    "--headless",
], check=True)

if not preview_path.exists():
    print("Demo preview not found. Generating fallback preview frame...")
    import pygame
    from environment import CarRacingEnv
    from utils import init_pygame

    init_pygame(headless=True)
    env = CarRacingEnv(headless=True, enable_metrics=False)
    env.reset()
    env.render()
    pygame.image.save(env.screen, str(preview_path))
    env.close()
    pygame.quit()

if preview_path.exists():
    print(f"Preview available: {preview_path}")
    display(Image(filename=str(preview_path)))
else:
    raise FileNotFoundError("Preview image was not generated.")

---
## 3–5. Experiment Batches

The full experiment grid has **9 experiments** (3 reward modes × 3 noise levels).

They are split into 3 batches of 3 to **avoid Colab timeouts**:

| Batch | Experiments | Index Range |
|-------|------------|-------------|
| Batch 1 | Experiments 1–3 | `--start-index 0` |
| Batch 2 | Experiments 4–6 | `--start-index 3` |
| Batch 3 | Experiments 7–9 | `--start-index 6` |

> **Important:** Run each batch cell one at a time. If you get disconnected, just reconnect and skip to the next unfinished batch.

### Optional: Live Log Monitor (Run in Parallel)

Use this while batches are running to confirm logs are being written in real time.
Stop this cell anytime with the interrupt button.

In [ ]:
import os
import time

print("Starting log monitor...")
print("Press the stop button to end monitoring.")

try:
    while True:
        print("\n--- Checking logs ---")
        os.system("ls -lah logs")
        time.sleep(30)
except KeyboardInterrupt:
    print("Log monitor stopped by user.")

### Batch 1 — Experiments 1–3

In [ ]:
import subprocess
import sys

print("===== BATCH 1: Experiments 1-3 =====")
print("Starting training...")
print("Running experiment batch 1 in headless mode...")
print("Command: python -u run_experiments.py --max-experiments 3 --start-index 0 --headless")

process = subprocess.Popen(
    [
        sys.executable, "-u", "run_experiments.py",
        "--max-experiments", "3",
        "--start-index", "0",
        "--headless",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("Saving logs...")

if return_code == 0:
    print("Batch 1 completed successfully")
else:
    print(f"Batch 1 failed with return code {return_code}")
    print("Error detected. Check the streamed logs above and re-run this batch.")
    raise RuntimeError(f"Batch 1 failed (return code: {return_code})")

### Batch 2 — Experiments 4–6

In [ ]:
import subprocess
import sys

print("===== BATCH 2: Experiments 4-6 =====")
print("Starting training...")
print("Running experiment batch 2 in headless mode...")
print("Command: python -u run_experiments.py --max-experiments 3 --start-index 3 --headless")

process = subprocess.Popen(
    [
        sys.executable, "-u", "run_experiments.py",
        "--max-experiments", "3",
        "--start-index", "3",
        "--headless",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("Saving logs...")

if return_code == 0:
    print("Batch 2 completed successfully")
else:
    print(f"Batch 2 failed with return code {return_code}")
    print("Error detected. Check the streamed logs above and re-run this batch.")
    raise RuntimeError(f"Batch 2 failed (return code: {return_code})")

### Batch 3 — Experiments 7–9

In [ ]:
import subprocess
import sys

print("===== BATCH 3: Experiments 7-9 =====")
print("Starting training...")
print("Running experiment batch 3 in headless mode...")
print("Command: python -u run_experiments.py --max-experiments 3 --start-index 6 --headless")

process = subprocess.Popen(
    [
        sys.executable, "-u", "run_experiments.py",
        "--max-experiments", "3",
        "--start-index", "6",
        "--headless",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("Saving logs...")

if return_code == 0:
    print("Batch 3 completed successfully")
else:
    print(f"Batch 3 failed with return code {return_code}")
    print("Error detected. Check the streamed logs above and re-run this batch.")
    raise RuntimeError(f"Batch 3 failed (return code: {return_code})")

### Verify Experiment Outputs

Quick check that all expected experiment directories and log files were created.

In [ ]:
from pathlib import Path

logs_dir = Path("logs")
models_dir = Path("models")

print("Experiment output verification")
print("=" * 40)

if logs_dir.exists():
    log_subdirs = sorted([p.name for p in logs_dir.iterdir() if p.is_dir()])
    print(f"Log directories ({len(log_subdirs)}): {log_subdirs}")
    
    # Check for training_log.jsonl in each
    for d in log_subdirs:
        log_file = logs_dir / d / "training_log.jsonl"
        status = "OK" if log_file.exists() else "MISSING"
        print(f"  {d}/training_log.jsonl ... {status}")
else:
    print("WARNING: logs/ directory not found!")

print()

if models_dir.exists():
    model_subdirs = sorted([p.name for p in models_dir.iterdir() if p.is_dir()])
    print(f"Model directories ({len(model_subdirs)}): {model_subdirs}")
else:
    print("WARNING: models/ directory not found!")

---
## 6. Plot Generation

Generate publication-ready plots from all experiment logs.

This produces:
- **Per-experiment plots**: reward vs episodes, crash rate, laps completed
- **Comparison plots**: all experiments overlaid for direct comparison

In [ ]:
import subprocess
import sys

print("Generating comparison plots for all experiments...")
print("=" * 50)

result = subprocess.run([
    sys.executable, "plot_metrics.py",
    "--log-dir", "logs",
    "--compare",
])

if result.returncode == 0:
    print()
    print("All plots generated successfully!")
else:
    print(f"\nPlot generation returned code {result.returncode}")
    print("Some plots may be missing. Check logs above.")

### Optional: Plot a Specific Experiment

Change the experiment name below to plot any individual experiment:

Available experiment names follow the pattern: `basic_noise_0.00`, `shaped_noise_0.02`, `modified_noise_0.05`, etc.

In [ ]:
import subprocess
import sys

# Change this to plot a specific experiment
EXPERIMENT_TAG = "R1_N1"  # e.g., R1_N1, R2_N2, R3_N3, etc.

print(f"Generating plots for experiment: {EXPERIMENT_TAG}")

subprocess.run([
    sys.executable, "plot_metrics.py",
    "--log-dir", "logs",
    "--experiments", EXPERIMENT_TAG,
])

### Display Generated Plots Inline

Show the comparison plots directly in the notebook.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

comparison_dir = Path("logs/comparison")

if comparison_dir.exists():
    png_files = sorted(comparison_dir.glob("*.png"))
    if png_files:
        print(f"Found {len(png_files)} comparison plot(s):")
        for png in png_files:
            print(f"  - {png.name}")
            display(Image(filename=str(png), width=800))
    else:
        print("No PNG files found in logs/comparison/")
else:
    print("Comparison directory not found. Run the plot generation cell first.")

# Also show any per-experiment plots
logs_dir = Path("logs")
exp_pngs = sorted(logs_dir.rglob("*.png"))
exp_pngs = [p for p in exp_pngs if "comparison" not in str(p) and "demo_preview" not in p.name]

if exp_pngs:
    print(f"\nFound {len(exp_pngs)} per-experiment plot(s).")
    # Show first few to avoid overwhelming output
    for png in exp_pngs[:6]:
        print(f"  - {png.relative_to(logs_dir)}")
        display(Image(filename=str(png), width=700))

---
## 7. Save Results — Organize Outputs

Collect all experiment outputs (logs, models, plots) into a single `results/` folder.

In [ ]:
import os
import shutil
from pathlib import Path

print("Organizing results...")
print("=" * 40)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Copy logs
if Path("logs").exists():
    shutil.copytree("logs", str(results_dir / "logs"), dirs_exist_ok=True)
    print("Copied: logs/ -> results/logs/")
else:
    print("WARNING: logs/ not found — skipping")

# Copy models
if Path("models").exists():
    shutil.copytree("models", str(results_dir / "models"), dirs_exist_ok=True)
    print("Copied: models/ -> results/models/")
else:
    print("WARNING: models/ not found — skipping")

# Copy any top-level PNG plots
png_count = 0
for file in Path(".").iterdir():
    if file.suffix == ".png" and file.is_file():
        shutil.copy2(str(file), str(results_dir / file.name))
        png_count += 1

if png_count > 0:
    print(f"Copied: {png_count} top-level PNG file(s) -> results/")

# Summary
total_files = sum(1 for _ in results_dir.rglob("*") if _.is_file())
print(f"\nResults organized: {total_files} files in results/")

---
## 8. Download Results

Zip everything in `results/` and download it.

> **Note:** The `files.download()` call only works in Google Colab. In a local environment, the ZIP file will be created but not auto-downloaded.

In [ ]:
import shutil
from pathlib import Path

results_dir = Path("results")
zip_name = "td3_results"

if not results_dir.exists() or not any(results_dir.rglob("*")):
    print("ERROR: results/ is empty or missing. Run the 'Save Results' cell first.")
else:
    print("Creating ZIP archive...")
    archive_path = shutil.make_archive(zip_name, "zip", str(results_dir))
    zip_size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
    print(f"Archive created: {archive_path} ({zip_size_mb:.1f} MB)")

    # Auto-download on Colab
    try:
        from google.colab import files
        print("Downloading via Colab...")
        files.download(archive_path)
    except ImportError:
        print("Not running in Colab — ZIP file saved locally.")
        print(f"Find it at: {Path(archive_path).resolve()}")